# 04 — Model regresji liniowej

## Cel

W tym notebooku buduję prosty model regresji liniowej, którego zadaniem
jest przewidywanie oprocentowania pożyczki (`interest_rate`).

Kolejne kroki to:
- podział danych na cechy `X` i target `y`,
- podział danych na zbiór treningowy i testowy,
- wytrenowanie modelu `LinearRegression`,
- wykonanie predykcji,
- ocena modelu za pomocą `R²` i `RMSE`,
- prosta interpretacja współczynników modelu.

## 1. Import bibliotek i wczytanie danych

In [14]:
# Path służy do tworzenia i obsługi ścieżek do plików i folderów.
from pathlib import Path

# NumPy wykorzystam do prostych obliczeń numerycznych.
import numpy as np

# Pandas służy do pracy z danymi tabelarycznymi.
import pandas as pd

# LinearRegression to model regresji liniowej,
# którego użyję do przewidywania oprocentowania.
from sklearn.linear_model import LinearRegression

# Metryki służą do oceny jakości predykcji modelu.
from sklearn.metrics import mean_squared_error, r2_score

# train_test_split służy do podziału danych
# na zbiór treningowy i testowy.
from sklearn.model_selection import train_test_split

In [15]:
# Ścieżka do danych przygotowanych wcześniej do modelu.
DATA_PATH = Path("data/03_do_modelu.csv")

# Nazwa kolumny, którą chcę przewidywać.
TARGET = "interest_rate"

# Wczytuję dane przygotowane w notebooku 02_transformacja.
df = pd.read_csv(DATA_PATH)

# Sprawdzam rozmiar danych.
print("Kształt:", df.shape)

# Wyświetlam pierwsze 5 wierszy,
# żeby sprawdzić, czy plik został poprawnie wczytany.
df.head()

Kształt: (10000, 10)


,loan_amount,term,annual_income,debt_to_income,emp_length,interest_rate,verified_income_Source Verified,verified_income_Verified,homeownership_OWN,homeownership_RENT
0,28000,60,90000.0,18.01,3.0,14.07,0,1,0,0
1,5000,36,40000.0,5.04,10.0,12.61,0,0,0,1
2,2000,36,40000.0,21.15,3.0,17.09,1,0,0,1
3,21600,36,30000.0,10.16,1.0,6.72,0,0,0,1
4,23000,36,35000.0,57.96,10.0,14.07,0,1,0,1


### Wnioski po wczytaniu danych do modelu

Dataset przygotowany do modelu zawiera 10 000 wierszy i 10 kolumn.

Liczba wierszy pozostała bez zmian, natomiast liczba kolumn jest mniejsza niż
w oryginalnym datasecie, ponieważ do modelu wybrałem tylko kilka cech.

Zmienne tekstowe zostały wcześniej zakodowane do postaci numerycznej,
więc dane są gotowe do podziału na cechy `X` i target `y`.

## 2. Podział danych na X i y

Model regresji liniowej potrzebuje osobno:

- `X` — cech wejściowych, czyli informacji używanych do przewidywania,
- `y` — targetu, czyli wartości, którą model ma przewidzieć.

W tym projekcie targetem jest `interest_rate`.

Dlatego do `X` zapisuję wszystkie kolumny poza targetem,
a do `y` tylko kolumnę `interest_rate`.

In [16]:
# X zawiera wszystkie cechy używane do przewidywania.
# Usuwam z niego kolumnę targetową, ponieważ model nie może
# używać odpowiedzi jako jednej z cech wejściowych.
X = df.drop(columns=[TARGET])

# y zawiera target, czyli oprocentowanie,
# które model będzie próbował przewidzieć.
y = df[TARGET]

# Sprawdzam rozmiary X i y.
print("Kształt X:", X.shape)
print("Kształt y:", y.shape)


Kształt X: (10000, 9)
Kształt y: (10000,)


### Wnioski

Zbiór `X` zawiera 9 cech wejściowych, a `y` zawiera 10 000 wartości targetu.

Dane są teraz rozdzielone na część wejściową i wartość przewidywaną,
więc można przejść do podziału na zbiór treningowy i testowy.

## 3. Podział danych na zbiór treningowy i testowy

Dane dzielę na dwie części:

- zbiór treningowy — na nim model będzie się uczył,
- zbiór testowy — posłuży do sprawdzenia jakości modelu na danych,
  których wcześniej nie widział.

Używam podziału 80% danych do treningu i 20% do testu.

In [17]:
# Dzielę dane na zbiór treningowy i testowy.
#
# test_size=0.2 oznacza, że 20% danych trafi do zbioru testowego,
# a pozostałe 80% do treningowego.
#
# random_state=42 zapewnia powtarzalność podziału,
# czyli przy kolejnym uruchomieniu otrzymam taki sam podział danych.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Sprawdzam rozmiary utworzonych zbiorów.

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (8000, 9)
X_test: (2000, 9)
y_train: (8000,)
y_test: (2000,)


### Wnioski

Dane zostały podzielone w proporcji 80/20.

Model będzie trenowany na 8 000 obserwacji,
a jego jakość zostanie sprawdzona na 2 000 obserwacji testowych.

## 4. Trenowanie modelu

Tworzę model regresji liniowej, a następnie uczę go na zbiorze treningowym.

Model będzie próbował znaleźć zależność pomiędzy cechami `X_train`
a targetem `y_train`.

In [18]:
# Tworzę obiekt modelu regresji liniowej.
model = LinearRegression()

# Uczę model na danych treningowych.
# X_train zawiera cechy, a y_train wartości targetu.
model.fit(X_train, y_train)

# Sprawdzam wyraz wolny modelu.
print("Wyraz wolny:", model.intercept_)

# Sprawdzam współczynniki modelu.
print("Współczynniki:", model.coef_)


Wyraz wolny: 4.024211765412174
Współczynniki: [-4.88038205e-05  1.68384476e-01 -4.23123066e-06  3.40777919e-02
 -2.42416326e-02  1.31397789e+00  2.87267400e+00  4.82342494e-01
  1.22750596e+00]


### Wnioski

Model został wytrenowany na 8 000 obserwacji treningowych.

Na tym etapie model ma już wyznaczone współczynniki regresji,
ale jego jakość sprawdzę dopiero na zbiorze testowym.

## 5. Predykcja i ocena modelu

Po wytrenowaniu modelu wykonuję predykcję dla zbioru testowego.

Następnie porównuję wartości przewidziane przez model z rzeczywistymi
wartościami `interest_rate`.

Do oceny modelu wykorzystuję dwie metryki:

- `R²` — informuje, jaką część zmienności targetu model potrafi wyjaśnić,
- `RMSE` — pokazuje typowy rozmiar błędu predykcji w tej samej jednostce co target.

In [19]:
# Model przewiduje wartości interest_rate
# dla obserwacji ze zbioru testowego.
y_pred = model.predict(X_test)

# R² porównuje wartości rzeczywiste z przewidywanymi
# i informuje, jak dobrze model wyjaśnia zmienność targetu.
R2 = r2_score(y_test, y_pred)

# Najpierw obliczam MSE, czyli średni kwadrat błędu,
# a następnie pierwiastek z MSE, aby otrzymać RMSE.
MSE = mean_squared_error(y_test, y_pred)
RMSE = np.sqrt(MSE)

print(f"R²: {R2:.3f}")
print(f"RMSE: {RMSE:.3f}")

R²: 0.227
RMSE: 4.528


In [20]:
# Tworzę małą tabelę porównującą
# rzeczywiste i przewidywane wartości oprocentowania.

porownanie = pd.DataFrame({
    "rzeczywiste": y_test,
    "przewidywane": y_pred
})

porownanie.head(10)

,rzeczywiste,przewidywane
6252,6.08,13.849742
4684,15.04,12.192769
1731,15.04,9.704995
4742,7.96,13.526179
4521,13.59,11.925417
6340,10.42,11.031063
576,12.62,11.708347
5202,17.09,12.964054
6363,5.32,9.448857
439,17.09,14.665553


In [21]:
# Obliczam błąd pojedynczej predykcji.
# Wartość dodatnia oznacza, że model zawyżył oprocentowanie,
# a ujemna, że je zaniżył.
porownanie["blad"] = (
    porownanie["przewidywane"]
    - porownanie["rzeczywiste"]
)

# Wartość bezwzględna pokazuje wielkość błędu
# bez względu na jego kierunek.
porownanie["blad_bezwzgledny"] = (
    porownanie["blad"].abs()
)

porownanie.head(10)

,rzeczywiste,przewidywane,blad,blad_bezwzgledny
6252,6.08,13.849742,7.769742,7.769742
4684,15.04,12.192769,-2.847231,2.847231
1731,15.04,9.704995,-5.335005,5.335005
4742,7.96,13.526179,5.566179,5.566179
4521,13.59,11.925417,-1.664583,1.664583
6340,10.42,11.031063,0.611063,0.611063
576,12.62,11.708347,-0.911653,0.911653
5202,17.09,12.964054,-4.125946,4.125946
6363,5.32,9.448857,4.128857,4.128857
439,17.09,14.665553,-2.424447,2.424447


### Przykładowe predykcje

Porównanie wartości rzeczywistych i przewidywanych pokazuje, że część
predykcji jest stosunkowo bliska wartościom rzeczywistym, ale występują
również znaczne błędy.

Model ma tendencję do przewidywania wartości bliższych średniemu
oprocentowaniu. Niskie wartości bywają zawyżane, a wysokie zaniżane.

Tabela pokazuje jednak tylko kilka przykładowych obserwacji.
Do oceny jakości całego modelu służą przede wszystkim metryki `R²` i `RMSE`.

## 6. Interpretacja współczynników i wyników modelu

Regresja liniowa przypisuje każdej cesze współczynnik, który pokazuje,
w jaki sposób zmiana danej cechy wiąże się ze zmianą przewidywanego
oprocentowania.

Dodatni współczynnik oznacza wzrost przewidywanego `interest_rate`,
a ujemny jego spadek, przy założeniu że pozostałe cechy się nie zmieniają.

In [22]:
# Tworzę tabelę ze współczynnikami modelu.
# Dzięki temu łatwiej sprawdzić, które cechy mają dodatni,
# a które ujemny związek z przewidywanym oprocentowaniem.

wspolczynniki = pd.DataFrame({
    "cecha": X.columns,
    "wspolczynnik": model.coef_
})

wspolczynniki = wspolczynniki.sort_values(
    "wspolczynnik",
    ascending=False
)

wspolczynniki


,cecha,wspolczynnik
6,verified_income_Verified,2.872674
5,verified_income_Source Verified,1.313978
8,homeownership_RENT,1.227506
7,homeownership_OWN,0.482342
1,term,0.168384
3,debt_to_income,0.034078
2,annual_income,-0.000004
0,loan_amount,-0.000049
4,emp_length,-0.024242


## 7. Wnioski


Pierwszy model regresji liniowej działa, ale jego jakość jest na razie
dość słaba.

Wynik `R² = 0.227` oznacza, że model wyjaśnia około 22.7% zmienności
oprocentowania. Pozostała część zależy od informacji, których obecny
zestaw cech nie uwzględnia, albo od zależności, których prosty model
liniowy nie potrafi dobrze odwzorować.

Porównanie wartości rzeczywistych i przewidywanych również pokazuje,
że model często przewiduje wartości bliższe średniemu oprocentowaniu.
W przypadku części pożyczek różnice są niewielkie, ale zdarzają się też
znacznie większe błędy.

Model traktuję więc jako pierwszy, prosty model bazowy, a nie rozwiązanie
końcowe. W dalszej pracy warto przede wszystkim popracować nad doborem
cech, sprawdzić dodatkowe informacje dostępne w datasecie oraz porównać
wyniki z innymi modelami regresyjnymi.

Na potrzeby tego projektu udało się jednak przejść cały proces:
od czyszczenia i transformacji danych, przez EDA, aż do zbudowania
i oceny modelu regresji liniowej.

## CDN.